# ECSC Developmental Sentence Contrast Analysis

**Question:** Does long-range coherence structure in spontaneous child speech change with age?

**Prediction:** Younger children (smaller working memory capacity) should show:
- Steeper decay of the contrast curve (less benefit from distant context)
- Lower absolute contrast scores (less coherent narratives overall)

**Method:** Same sentence contrast approach as RAID v5 — for each sentence in the narrative, compare model's perplexity on the real next sentence vs impostor sentences from other children's narratives of the same story. Impostors are matched by story (same frog story) to control for content/topic.

**Data:** ECSC (Eugene Children's Story Corpus) — 341 frog story narrations from children aged 5-11 years. Spontaneous speech, no pre-planning — the strongest test case for the memory hypothesis.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/ecsc_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/ECSC_contrast")
    if (DRIVE_DATA / "transcripts.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/ecsc_processed")
        if not (LOCAL_DATA / "transcripts.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload transcripts.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/ecsc_contrast")
    DATA_DIR = Path("../data/ecsc_processed")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

WINDOWS = [4, 16, 32, 64, 128]
N_IMPOSTORS = 2
RANDOM_SEED = 42
MIN_WORDS = 150  # minimum transcript length
MAX_TARGET_SENTS = 3  # limit sentence boundaries per doc for speed

# Age bins for analysis
AGE_BINS = [
    (59, 78, '5-6y'),
    (79, 96, '7-8y'),
    (97, 138, '9-11y'),
]

print(f"Windows: {WINDOWS}")
print(f"Age bins: {[b[2] for b in AGE_BINS]}")

In [ ]:
# Load corpus and extract metadata
corpus = []
with open(DATA_DIR / "transcripts.jsonl") as f:
    for line in f:
        d = json.loads(line)
        pop = json.loads(d['population'])
        d['age_months'] = pop['age_months']
        d['sex'] = pop['sex']
        d['study_year'] = pop['study_year']
        d['word_count'] = len(d['text'].split())
        # Extract story name from doc_id
        doc_id = d['doc_id']
        d['story'] = 'unknown'
        for story in ['one_frog_too_many', 'frog_where_are_you', 'boy_dog_frog', 'frog_on_his_own']:
            # Story is encoded in the original filename, approximate from text length/content
            pass
        corpus.append(d)

# Filter to minimum length
corpus = [d for d in corpus if d['word_count'] >= MIN_WORDS]
print(f"Loaded {len(corpus)} transcripts (>= {MIN_WORDS} words)")

# Build impostor pools by age bin
# Impostors come from OTHER children in the SAME age bin
impostor_pools = {}
for lo, hi, label in AGE_BINS:
    pool_docs = [d for d in corpus if lo <= d['age_months'] <= hi]
    pool_sents = {}
    for d in pool_docs:
        sents = re.split(r'(?<=[.!?])\s+', d['text'].strip())
        sents = [s.strip() for s in sents if len(s.strip().split()) >= 4]
        if len(sents) >= 3:
            pool_sents[d['doc_id']] = sents
    impostor_pools[label] = pool_sents
    n_sents = sum(len(v) for v in pool_sents.values())
    print(f"  {label}: {len(pool_docs)} docs, {n_sents} sentences in impostor pool")

# Assign age bin to each doc
for d in corpus:
    for lo, hi, label in AGE_BINS:
        if lo <= d['age_months'] <= hi:
            d['age_bin'] = label
            break

print(f"\nBy age bin:")
for lo, hi, label in AGE_BINS:
    n = sum(1 for d in corpus if d.get('age_bin') == label)
    ages = [d['age_months'] for d in corpus if d.get('age_bin') == label]
    print(f"  {label}: n={n}, age range={min(ages)}-{max(ages)} months")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_ppl_on_tokens(context_ids, target_ids):
    """Compute perplexity of target_ids given context_ids as prefix."""
    full_ids = context_ids + target_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    target_start = len(context_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def split_into_sentences(text, tokenizer):
    """Split text into sentences with token positions."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    result = []
    current_pos = 0
    for sent_text in sentences:
        sent_ids = tokenizer.encode(sent_text, add_special_tokens=False)
        if len(sent_ids) >= 3:
            result.append({
                'text': sent_text,
                'ids': sent_ids,
                'start': current_pos,
                'end': current_pos + len(sent_ids),
            })
        current_pos += len(sent_ids)
    return result


def get_impostor_sentences(age_bin, exclude_doc_id, rng, n=5):
    """Get impostor sentences from OTHER children in the same age bin."""
    pool = impostor_pools.get(age_bin, {})
    candidates = []
    for doc_id, sents in pool.items():
        if doc_id != exclude_doc_id:
            candidates.extend(sents)
    if len(candidates) < n:
        return []
    chosen = rng.choice(candidates, size=n, replace=False)
    return list(chosen)


def compute_contrast_curve(doc, rng):
    """Compute sentence contrast scores at each context window."""
    text = doc['text']
    age_bin = doc['age_bin']
    doc_id = doc['doc_id']
    full_ids = tokenizer.encode(text, add_special_tokens=False)
    n_tokens = len(full_ids)

    if n_tokens < 100:
        return None

    sentences = split_into_sentences(text, tokenizer)
    if len(sentences) < 4:
        return None

    mid = len(sentences) // 2
    target_sents = sentences[mid:]
    if len(target_sents) < 2:
        return None
    if len(target_sents) > MAX_TARGET_SENTS:
        idx = rng.choice(len(target_sents), MAX_TARGET_SENTS, replace=False)
        target_sents = [target_sents[j] for j in idx]

    results_by_W = {}

    for W in WINDOWS:
        contrast_scores = []
        for tgt in target_sents:
            sent_start = tgt['start']
            sent_end = tgt['end']
            if sent_end > n_tokens or sent_start < W:
                continue
            context_start = max(0, sent_start - W)
            context_ids = full_ids[context_start:sent_start]
            real_sent_ids = tgt['ids']
            if len(real_sent_ids) < 3 or len(context_ids) < 2:
                continue

            ppl_real = compute_ppl_on_tokens(context_ids, real_sent_ids)
            if math.isinf(ppl_real) or ppl_real <= 0:
                continue

            imp_texts = get_impostor_sentences(age_bin, doc_id, rng, n=N_IMPOSTORS)
            if len(imp_texts) < 2:
                continue

            ppl_impostors = []
            for imp_text in imp_texts:
                imp_ids = tokenizer.encode(imp_text, add_special_tokens=False)
                if len(imp_ids) < 3:
                    continue
                ppl_imp = compute_ppl_on_tokens(context_ids, imp_ids)
                if not math.isinf(ppl_imp) and ppl_imp > 0:
                    ppl_impostors.append(ppl_imp)

            if len(ppl_impostors) < 2:
                continue

            contrast = math.log(np.mean(ppl_impostors) / ppl_real)
            contrast_scores.append(contrast)

        if len(contrast_scores) >= 2:
            results_by_W[W] = {
                'contrast_mean': np.mean(contrast_scores),
                'contrast_std': np.std(contrast_scores),
                'n_sentences': len(contrast_scores),
            }

    return results_by_W if len(results_by_W) >= 3 else None


def extract_row(doc, curve, n_tokens):
    row = {
        'doc_id': doc['doc_id'],
        'author_id': doc['author_id'],
        'age_months': doc['age_months'],
        'age_bin': doc['age_bin'],
        'sex': doc['sex'],
        'word_count': doc['word_count'],
        'token_count': n_tokens,
    }
    for W, metrics in sorted(curve.items()):
        row[f'contrast_W{W}'] = metrics['contrast_mean']
        row[f'n_sent_W{W}'] = metrics['n_sentences']
    return row


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "ecsc_contrast_results.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Transcripts"):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < 100:
            skipped += 1
            continue
        curve = compute_contrast_curve(doc, rng)
        if curve is None:
            skipped += 1
            continue
        results.append(extract_row(doc, curve, len(token_ids)))

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} transcripts ({skipped} skipped), saved to {results_path}")

print(f"\nBy age bin:")
for lo, hi, label in AGE_BINS:
    n = len(df[df.age_bin == label])
    print(f"  {label}: n={n}")

In [ ]:
contrast_cols = [f'contrast_W{w}' for w in WINDOWS]
age_colors = {'5-6y': '#e74c3c', '7-8y': '#f39c12', '9-11y': '#27ae60'}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Raw contrast by age bin
ax = axes[0, 0]
for lo, hi, label in AGE_BINS:
    sub = df[df.age_bin == label]
    vals = np.array([sub[c].mean() for c in contrast_cols if c in sub.columns])
    sems = np.array([sub[c].sem() for c in contrast_cols if c in sub.columns])
    ax.plot(WINDOWS[:len(vals)], vals, 'o-', color=age_colors[label], linewidth=2, markersize=5, label=f'{label} (n={len(sub)})')
    ax.fill_between(WINDOWS[:len(vals)], vals - sems, vals + sems, color=age_colors[label], alpha=0.15)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Contrast Score\nlog(ppl_impostor / ppl_real)')
ax.set_title('A. Sentence Contrast by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Normalized curves by age bin
ax = axes[0, 1]
for lo, hi, label in AGE_BINS:
    sub = df[df.age_bin == label]
    vals = np.array([sub[c].mean() for c in contrast_cols if c in sub.columns])
    total = vals[-1] - vals[0]
    if total > 0.01:
        norm = (vals - vals[0]) / total
        ax.plot(WINDOWS[:len(norm)], norm, 'o-', color=age_colors[label], linewidth=2, markersize=5, label=label)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2, label='Linear')
ax.set_xscale('log', base=2)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Fraction of Total Contrast Gain')
ax.set_title('B. Normalized Contrast Curves by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# C: Marginal gain per token by age
ax = axes[1, 0]
for lo, hi, label in AGE_BINS:
    sub = df[df.age_bin == label]
    vals = np.array([sub[c].mean() for c in contrast_cols if c in sub.columns])
    marg, marg_x = [], []
    for i in range(1, len(vals)):
        dw = WINDOWS[i] - WINDOWS[i-1]
        marg.append((vals[i] - vals[i-1]) / dw)
        marg_x.append((WINDOWS[i] + WINDOWS[i-1]) / 2)
    ax.plot(marg_x, marg, 'o-', color=age_colors[label], linewidth=1.5, markersize=4, label=label)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Distance (tokens)')
ax.set_ylabel('Marginal Contrast Gain per Token')
ax.set_title('C. Influence Decay by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# D: Age as continuous predictor — contrast at W128 vs age_months
ax = axes[1, 1]
for w_plot in [32, 64, 128]:
    col = f'contrast_W{w_plot}'
    if col in df.columns:
        valid = df[[col, 'age_months']].dropna()
        ax.scatter(valid['age_months'], valid[col], alpha=0.3, s=15, label=f'W{w_plot}')
        # Regression line
        slope, intercept, r, p, se = stats.linregress(valid['age_months'], valid[col])
        x_line = np.array([valid['age_months'].min(), valid['age_months'].max()])
        ax.plot(x_line, intercept + slope * x_line, '--', linewidth=2,
                label=f'W{w_plot}: r={r:.3f}, p={p:.4f}')

ax.set_xlabel('Age (months)')
ax.set_ylabel('Contrast Score')
ax.set_title('D. Contrast vs Age (continuous)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

plt.suptitle('Developmental Sentence Contrast: Does Long-Range Coherence Increase with Age?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_developmental.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary stats
print("\nContrast by age bin:")
print(f"{'Age':<8}", end='')
for w in WINDOWS:
    print(f"{'W'+str(w):>8}", end='')
print()
for lo, hi, label in AGE_BINS:
    sub = df[df.age_bin == label]
    print(f"{label:<8}", end='')
    for w in WINDOWS:
        col = f'contrast_W{w}'
        print(f"{sub[col].mean():>8.3f}" if col in sub.columns else f"{'N/A':>8}", end='')
    print()

print("\nAge correlation with contrast (continuous):")
for w in WINDOWS:
    col = f'contrast_W{w}'
    if col in df.columns:
        valid = df[[col, 'age_months']].dropna()
        r, p = stats.pearsonr(valid['age_months'], valid[col])
        print(f"  W{w:<4}: r={r:+.3f}, p={p:.4f}")